# Site pipeline

The same Python the website runs, step by step. Run from the repo root (or keep the `sys.path` line) after `pip install -r requirements-dev.txt`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from billbreaker.ml.notebook_run import EXAMPLE_PROFILE, EXAMPLE_RUN
from billbreaker.ml.profile_terms import clean_profile, profile_terms
from billbreaker.ml.sections import rank_sections
from billbreaker.ml.claims_check import FLAG_LABELS, check_claims, serious_count

## 1. Reader answers to search terms

The eight optional questions in the search dialog become the words the ranker looks for.

In [2]:
clean_profile(EXAMPLE_PROFILE)

{'state': 'Colorado',
 'age': '18-24',
 'work': 'college student, part-time retail',
 'household': 'just me',
 'health': "on my parents' employer plan",
 'income': 'under $30k',
 'interests': 'environment, student loans, public lands'}

In [3]:
profile_terms(EXAMPLE_PROFILE)

['assistance',
 'bureau of land management',
 'climate',
 'college',
 'colorado',
 'conservation',
 'credit',
 'education',
 'employer',
 'environment',
 'group health plan',
 'lands',
 'loan',
 'loans',
 'low-income',
 'national forest',
 'part-time',
 'pollution',
 'poverty',
 'public land',
 'retail',
 'school',
 'student',
 'tuition',
 'university',
 'wilderness',
 'wildlife',
 'young',
 'youth']

## 2. Rank a bill's sections for that reader

Paste any bill text with `SEC.` headings into `BILL`.

In [4]:
BILL = """SEC. 1. SHORT TITLE.
This Act may be cited as the Example Act.
SEC. 2. STUDENT LOANS.
A student with loans from a college or university may defer tuition loan payments.
SEC. 3. PUBLIC LANDS.
The Bureau of Land Management shall protect wildlife on public land in Colorado.
SEC. 4. HIGHWAYS.
Funds for highways."""

ranking = rank_sections(BILL, EXAMPLE_PROFILE)
[(r["section"], r["heading"], r["score"], r["matchedTerms"]) for r in ranking["ranked"]]

[('2',
  'STUDENT LOANS',
  10,
  ['college', 'loan', 'loans', 'student', 'tuition', 'university']),
 ('3',
  'PUBLIC LANDS',
  6,
  ['bureau of land management',
   'colorado',
   'lands',
   'public land',
   'wildlife'])]

## 3. Check a personalized explanation against its sources

Every sentence gets a support score and flags for invented topics, numbers, or opinions.

In [5]:
sources = [EXAMPLE_RUN["general"], *(s["heading"] for s in EXAMPLE_RUN["sections"])]
report = check_claims(EXAMPLE_RUN["personalized"], sources, " ".join(EXAMPLE_PROFILE.values()))
print(serious_count(report), "of", len(report["claims"]), "sentences need review")
for c in report["claims"]:
    labels = ", ".join(FLAG_LABELS[f] for f in c["flags"]) or "ok"
    print(f"{c['support']:.0%}  {labels:30}  {c['text'][:80]}")

4 of 13 sentences need review
29%  Loosely supported               This bill aims to improve our understanding and management of marine debris acro
75%  ok                              Identify where marine debris comes from.
67%  ok                              Determine its sources more accurately.
50%  ok                              Assess the impact of marine debris on our oceans and waterways.
100%  ok                              Prevent and reduce marine debris.
60%  ok                              Remove existing marine debris effectively.
60%  Opinion                         By focusing on marine debris that poses a threat to living marine resources and 
57%  ok                              As a college student working part-time at a retail job, your life might be affec
29%  Loosely supported               Environmental Awareness: The bill encourages awareness about marine debris, whic
20%  Loosely supported               Public Lands: Since you live near public lands, any ne

## 4. Read the AP politics feed

The news section on the homepage parses this feed. The cell below uses a tiny sample so it runs offline; swap in `await legislation_news()` to fetch the live feed.

In [6]:
from billbreaker.services.ap_news import parse_feed

SAMPLE = """<rss><channel>
<item><title>Senate passes budget bill - AP News</title><link>https://news.google.com/rss/articles/CBMiAbC123</link><pubDate>Fri, 17 Oct 2026 14:00:00 GMT</pubDate></item>
</channel></rss>"""
parse_feed(SAMPLE)

[{'title': 'Senate passes budget bill - AP News',
  'link': 'https://news.google.com/rss/articles/CBMiAbC123',
  'pubDate': 'Fri, 17 Oct 2026 14:00:00 GMT',
  'source': 'AP News'}]